In [1]:
import json
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

In [2]:
df = pd.read_csv("../data/mtm24_transliterated.csv")

df = df[
    [
        "original_cuneiform",
        "transliteration"
    ]
]

print(df.shape)

(190234, 2)


In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 171210
Val: 9512
Test: 9512


In [4]:
src_chars = set()

for text in df["original_cuneiform"]:
    src_chars.update(text)

tgt_chars = set()

for text in df["transliteration"]:
    tgt_chars.update(text)

In [5]:
src_vocab = {
    "<pad>":0,
    "<sos>":1,
    "<eos>":2,
    "<unk>":3
}

for c in sorted(src_chars):
    src_vocab[c] = len(src_vocab)

In [6]:
tgt_vocab = {
    "<pad>":0,
    "<sos>":1,
    "<eos>":2,
    "<unk>":3
}

for c in sorted(tgt_chars):
    tgt_vocab[c] = len(tgt_vocab)

In [7]:
idx2src = {
    v:k
    for k,v in src_vocab.items()
}

idx2tgt = {
    v:k
    for k,v in tgt_vocab.items()
}

In [8]:
class CuneiformDataset(Dataset):

    def __init__(
        self,
        dataframe,
        src_vocab,
        tgt_vocab
    ):

        self.df = dataframe.reset_index(drop=True)

        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.df)

    def encode_src(self,text):

        return [
            self.src_vocab.get(
                ch,
                self.src_vocab["<unk>"]
            )
            for ch in text
        ]

    def encode_tgt(self,text):

        encoded = [
            self.tgt_vocab["<sos>"]
        ]

        encoded.extend(
            [
                self.tgt_vocab.get(
                    ch,
                    self.tgt_vocab["<unk>"]
                )
                for ch in text
            ]
        )

        encoded.append(
            self.tgt_vocab["<eos>"]
        )

        return encoded

    def __getitem__(self,idx):

        row = self.df.iloc[idx]

        src = torch.tensor(
            self.encode_src(
                row["original_cuneiform"]
            ),
            dtype=torch.long
        )

        tgt = torch.tensor(
            self.encode_tgt(
                row["transliteration"]
            ),
            dtype=torch.long
        )

        return src, tgt

In [9]:
def collate_fn(batch):

    src_batch = [x[0] for x in batch]
    tgt_batch = [x[1] for x in batch]

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=0
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=0
    )

    return src_batch, tgt_batch

In [10]:
train_ds = CuneiformDataset(
    train_df,
    src_vocab,
    tgt_vocab
)

val_ds = CuneiformDataset(
    val_df,
    src_vocab,
    tgt_vocab
)

test_ds = CuneiformDataset(
    test_df,
    src_vocab,
    tgt_vocab
)

In [11]:
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [12]:
src, tgt = next(
    iter(train_loader)
)

print(src.shape)
print(tgt.shape)

torch.Size([64, 206])
torch.Size([64, 746])


In [13]:
sample_src, sample_tgt = train_ds[0]

decoded_src = "".join(
    idx2src[i.item()]
    for i in sample_src
)

decoded_tgt = "".join(
    idx2tgt[i.item()]
    for i in sample_tgt[1:-1]
)

print(decoded_src)
print()
print(decoded_tgt)

...𒆳𒌵𒈨𒌍...𒉻...𒐊...𒐞𒐗𒌋𒌋...
...𒅆𒄴𒇷o
𒉌𒌓𒈩𒁹𒈾𒁍𒋾𒂗𒌦𒈨𒌍𒋫𒋧𒀀𒉌


{KUR}-URI MEŠ {MUŠEN} 5 ME 88 ši-ih-li-kut piš {NA₄}-KIŠIB {m}-na-bu-ti EN.UN.MEŠ ta-lik a-ni 


In [14]:
sample_src, sample_tgt = train_ds[0]

print(len(sample_src))
print(len(sample_tgt))

50
96


In [15]:
import numpy as np

src_lengths = [
    len(train_ds[i][0])
    for i in range(5000)
]

tgt_lengths = [
    len(train_ds[i][1])
    for i in range(5000)
]

print("SOURCE")
print("mean", np.mean(src_lengths))
print("95%", np.percentile(src_lengths,95))
print("max", np.max(src_lengths))

print()

print("TARGET")
print("mean", np.mean(tgt_lengths))
print("95%", np.percentile(tgt_lengths,95))
print("max", np.max(tgt_lengths))

SOURCE
mean 63.995
95% 149.0
max 395

TARGET
mean 188.3526
95% 466.0
max 1196


In [16]:
MAX_SRC_LEN = 200
MAX_TGT_LEN = 300

filtered_df = df[
    (df["original_cuneiform"].str.len() <= MAX_SRC_LEN)
    &
    (df["transliteration"].str.len() <= MAX_TGT_LEN)
]

print(filtered_df.shape)

(152406, 2)
